# Python Mutation Testing (SuT: `classify_triangle`)

In this exercise, we explore **mutation testing** using [mutmut](https://mutmut.readthedocs.io/), a mutation testing tool for Python.

Mutation testing evaluates the quality of your test suite by introducing small changes (mutations) to your code and checking whether your tests detect them. A **killed mutant** means your tests caught the change; a **survived mutant** indicates a gap in your test coverage.

We'll use the familiar `classify_triangle` function as our System under Test (SuT) and learn how to:
1. Run mutation testing with `mutmut`
2. Interpret mutation testing results
3. Improve test suites to kill surviving mutants

## Check how to use `mutmut`

In [1]:
!pytest --version
!mutmut version

pytest 8.3.3
mutmut version 2.4.4


In [1]:
!pytest tests_triangle/test_classify_triangle.py --cov=classify_triangle --cov-branch

============================= test session starts ==============================
platform linux -- Python 3.10.20, pytest-8.3.3, pluggy-1.6.0
rootdir: /workspace/modules/03_mutation_testing/exercises
plugins: anyio-4.13.0, cov-5.0.0
collected 3 items                                                              

tests_triangle/test_classify_triangle.py ...                             [100%]

---------- coverage: platform linux, python 3.10.20-final-0 ----------
Name                                  Stmts   Miss Branch BrPart  Cover
-----------------------------------------------------------------------
tests_triangle/classify_triangle.py      20      4     12      4    75%
-----------------------------------------------------------------------
TOTAL                                    20      4     12      4    75%


============================== 3 passed in 0.01s ===============================


In [8]:
!rm -rf .mutmut-cache

In [9]:
!mutmut -h

Usage: mutmut [OPTIONS] COMMAND [ARGS]...

  Mutation testing system for Python.

Options:
  -h, --help  Show this message and exit.

Commands:
  apply       Apply a mutation on disk.
  html        Generate a HTML report of surviving mutants.
  junitxml    Show a mutation diff with junitxml format.
  result-ids  Print the IDs of the specified mutant classes (separated by...
  results     Print the results.
  run         Runs mutmut.
  show        Show a mutation diff.
  version     Show the version and exit.


In [10]:
!mutmut run -h

Usage: mutmut run [OPTIONS] [ARGUMENT]

  Runs mutmut. You probably want to start with just trying this. If you supply
  a mutation ID mutmut will check just this mutant.

Options:
  --paths-to-mutate TEXT
  --disable-mutation-types TEXT   Skip the given types of mutations.
  --enable-mutation-types TEXT    Only perform given types of mutations.
  --paths-to-exclude TEXT
  --runner TEXT
  --use-coverage
  --use-patch-file TEXT           Only mutate lines added/changed in the given
                                  patch file
  --rerun-all                     If you modified the test_command in the
                                  pre_mutation hook, the default test_command
                                  (specified by the "runner" option) will be
                                  executed if the mutant survives with your
                                  modified test_command.
  --tests-dir TEXT
  -m, --test-time-multiplier FLOAT
  -b, --test-time-base FLOAT
  -s, --swallow-output  

## Run `mutmut` and interprete the results

In [12]:
!mutmut run --paths-to-mutate tests_triangle/classify_triangle.py --tests-dir tests_triangle


- Mutation testing starting -

These are the steps:
1. A full test suite run will be made to make sure we
   can run the tests successfully and we know how long
   it takes (to detect infinite loops for example)
2. Mutants will be generated and checked

Results are stored in .mutmut-cache.
Print found mutants with `mutmut results`.

Legend for output:
🎉 Killed mutants.   The goal is for everything to end up in this bucket.
⏰ Timeout.          Test suite took 10 times as long as the baseline so were killed.
🤔 Suspicious.       Tests took a long time, but not long enough to be fatal.
🙁 Survived.         This means your tests need to be expanded.
🔇 Skipped.          Skipped.

1. Using cached time for baseline tests, to run baseline again delete the cache file

2. Checking mutants
⠴ 26/26  🎉 17  ⏰ 0  🤔 0  🙁 9  🔇 0


In [13]:
!mutmut results

To apply a mutant on disk:
    mutmut apply <id>

To show a mutant:
    mutmut show <id>


Survived 🙁 (9)

---- tests_triangle/classify_triangle.py (9) ----

3-5, 7-8, 14-15, 23-24


In [14]:
!mutmut show 1

--- tests_triangle/classify_triangle.py
+++ tests_triangle/classify_triangle.py
@@ -8,7 +8,7 @@
 """
 def classify_triangle(a: float, b: float, c: float) -> str:
     """Classify a triangle based on three side lengths."""
-    sides = sorted([a, b, c])
+    sides = None
     x, y, z = sides  # x <= y <= z
 
     if x <= 0:



In [15]:
!for i in $(seq 1 26); do \
    echo "============================"; \
    echo "mutant $i:"; \
    mutmut show $i 2>/dev/null | grep -E '^[+-][^+-]' || echo "(no diff or invalid id)"; \
    echo ""; \
done

mutant 1:
-    sides = sorted([a, b, c])
+    sides = None

mutant 2:
-    x, y, z = sides  # x <= y <= z
+    x, y, z = None  # x <= y <= z

mutant 3:
-    if x <= 0:
+    if x < 0:

mutant 4:
-    if x <= 0:
+    if x <= 1:

mutant 5:
-        return "invalid"  # Negative or zero length is not allowed
+        return "XXinvalidXX"  # Negative or zero length is not allowed

mutant 6:
-    if x + y <= z:
+    if x - y <= z:

mutant 7:
-    if x + y <= z:
+    if x + y < z:

mutant 8:
-        return "invalid"  # Violates triangle inequality
+        return "XXinvalidXX"  # Violates triangle inequality

mutant 9:
-    if x == y == z:
+    if x != y == z:

mutant 10:
-    if x == y == z:
+    if x == y != z:

mutant 11:
-        return "equilateral"
+        return "XXequilateralXX"

mutant 12:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = (x != y) or (y == z)

mutant 13:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = (x == y) or (y != z)

mutant 14:
-    is_

### ‼️ Before running another test, remove the cache.

In [16]:
!rm -rf .mutmut-cache

### Run `mutmut` with `disable/enagle-mutation-types` option.

In [18]:
!mutmut run --disable-mutation-types XXX --paths-to-mutate tests/classify_triangle.py --tests-dir tests_triangle

Usage: mutmut run [OPTIONS] [ARGUMENT]
Try 'mutmut run -h' for help.

Error: The following are not valid mutation types: XXX. Valid mutation types are: operator, keyword, number, name, string, fstring, argument, or_test, and_test, lambdef, expr_stmt, decorator, annassign


| Mutation Type | Description |
|---------------|------------|
| **operator** | Modifies arithmetic or comparison operators (e.g., `+` → `-`, `<` → `<=`). |
| **keyword** | Alters Python keywords that affect control flow (e.g., `break`, `continue`, `return`). |
| **number** | Changes numeric literals (e.g., `0` → `1`, `5` → `6`). |
| **name** | Replaces variable names or special constants such as `None`, `True`, or `False`. |
| **string** | Mutates string literals (e.g., modifies or injects characters). |
| **fstring** | Alters formatted string literals (f-strings). |
| **argument** | Modifies function call arguments (e.g., removing or altering arguments). |
| **or_test** | Changes logical `or` expressions (e.g., altering or simplifying `or` conditions). |
| **and_test** | Changes logical `and` expressions (e.g., altering or simplifying `and` conditions). |
| **lambdef** | Mutates lambda function definitions. |
| **expr_stmt** | Modifies standalone expression statements. |
| **decorator** | Alters or removes decorators applied to functions or classes. |
| **annassign** | Mutates annotated assignments (variables with type annotations). |

---

**Source:**  
Mutation type names are derived from the `mutmut` mutation engine implementation (see `node_mutation.py` in the official mutmut repository):  
https://github.com/boxed/mutmut

In [21]:
!mutmut run --disable-mutation-types string,name --paths-to-mutate tests_triangle/classify_triangle.py --tests-dir tests_triangle


- Mutation testing starting -

These are the steps:
1. A full test suite run will be made to make sure we
   can run the tests successfully and we know how long
   it takes (to detect infinite loops for example)
2. Mutants will be generated and checked

Results are stored in .mutmut-cache.
Print found mutants with `mutmut results`.

Legend for output:
🎉 Killed mutants.   The goal is for everything to end up in this bucket.
⏰ Timeout.          Test suite took 10 times as long as the baseline so were killed.
🤔 Suspicious.       Tests took a long time, but not long enough to be fatal.
🙁 Survived.         This means your tests need to be expanded.
🔇 Skipped.          Skipped.

mutmut cache is out of date, clearing it...
1. Running tests without mutations
⠋ Running...Done

2. Checking mutants
⠹ 19/19  🎉 14  ⏰ 0  🤔 0  🙁 5  🔇 0


In [22]:
!mutmut results

To apply a mutant on disk:
    mutmut apply <id>

To show a mutant:
    mutmut show <id>


Survived 🙁 (5)

---- tests_triangle/classify_triangle.py (5) ----

3-4, 6, 11-12


In [23]:
!for i in $(seq 1 19); do \
    echo "============================"; \
    echo "mutant $i:"; \
    mutmut show $i 2>/dev/null | grep -E '^[+-][^+-]' || echo "(no diff or invalid id)"; \
    echo ""; \
done

mutant 1:
-    sides = sorted([a, b, c])
+    sides = None

mutant 2:
-    x, y, z = sides  # x <= y <= z
+    x, y, z = None  # x <= y <= z

mutant 3:
-    if x <= 0:
+    if x < 0:

mutant 4:
-    if x <= 0:
+    if x <= 1:

mutant 5:
-    if x + y <= z:
+    if x - y <= z:

mutant 6:
-    if x + y <= z:
+    if x + y < z:

mutant 7:
-    if x == y == z:
+    if x != y == z:

mutant 8:
-    if x == y == z:
+    if x == y != z:

mutant 9:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = (x != y) or (y == z)

mutant 10:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = (x == y) or (y != z)

mutant 11:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = (x == y) and (y == z)

mutant 12:
-    is_isosceles = (x == y) or (y == z)
+    is_isosceles = None

mutant 13:
-    is_right = (x * x + y * y == z * z)
+    is_right = (x / x + y * y == z * z)

mutant 14:
-    is_right = (x * x + y * y == z * z)
+    is_right = (x * x - y * y == z * z)

mutant 15:
-    is

## 🎯 Exercise: Kill All Mutants

In this exercise, your goal is to design a **strong test suite** that kills all generated mutants of `classify_triangle.py`.

You are given an initial set of tests in `test_classify_triangle.py`.  
However, the current tests are not sufficient — some mutants survive.

### Your Task

1. Run mutation testing using `mutmut`.
2. Identify the survived mutants using: `mutmut results/show`
3. Analyze why each survived mutant was not detected.
4. Improve `test_classify_triangle.py` by adding meaningful test cases.
5. Re-run mutation testing and repeat until **all mutants are killed**.